In [2]:
import os
import time
import json
import re
import csv
import traceback
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def safe_filename(variant):
    return re.sub(r'[<>:"/\\|?*]', "_", variant)

def double_encode(variant):
    return quote(quote(variant, safe=""), safe="")

def extract_pseudomonas_infection_from_table(filename):
    with open(filename, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    result = {
        "percent_with_infection_variant": None,
        "n_variant": None,
        "percent_with_infection_two_cf": None,
        "n_two_cf": None,
        "average_age_variant": None,
        "average_age_two_cf": None
    }

    tables = soup.find_all("table")
    for table in tables:
        headers = table.find_all("thead")
        if not headers:
            continue
        header_text = headers[0].get_text()
        if "infected" in header_text.lower() and "variant" in header_text.lower():
            rows = table.find_all("tr")
            if len(rows) >= 3:
                header_cells = rows[0].find_all("td")
                if len(header_cells) >= 3:
                    match_variant = re.search(r"number of patients\s*=\s*([\d,]+)", header_cells[1].text)
                    match_two_cf = re.search(r"number of patients\s*=\s*([\d,]+)", header_cells[2].text)
                    if match_variant:
                        result["n_variant"] = int(match_variant.group(1).replace(",", ""))
                    if match_two_cf:
                        result["n_two_cf"] = int(match_two_cf.group(1).replace(",", ""))

                data_cells = rows[1].find_all("td")
                if len(data_cells) >= 3:
                    def parse_percent(text):
                        return int(re.sub(r"[^\d]", "", text)) if re.search(r"\d", text) else None
                    result["percent_with_infection_variant"] = parse_percent(data_cells[1].text)
                    result["percent_with_infection_two_cf"] = parse_percent(data_cells[2].text)

                if len(rows) >= 4:
                    age_cells = rows[3].find_all("td")
                    if len(age_cells) >= 3:
                        def parse_age(text):
                            return int(re.sub(r"[^\d]", "", text)) if re.search(r"\d", text) else None
                        result["average_age_variant"] = parse_age(age_cells[1].text)
                        result["average_age_two_cf"] = parse_age(age_cells[2].text)

            break

    return result

# Load variant names
df = pd.read_excel("CFTR2_25September2024.xlsx", sheet_name="CFTR2 variants by legacy name", skiprows=9)
variant_names = [v.strip() for v in df["Variant legacy name"].dropna().unique()]

# Load existing data
if os.path.exists("cftr2_pseudomonas_infection.json"):
    with open("cftr2_pseudomonas_infection.json", encoding="utf-8") as f:
        infection_data_all = json.load(f)
else:
    infection_data_all = {}

normalized_keys = {k.strip() for k in infection_data_all}
SAVE_FOLDER = "cftr2_variants"
os.makedirs(SAVE_FOLDER, exist_ok=True)

# Chrome setup
options = Options()
# options.add_argument("--headless=new")  # Uncomment when stable
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# Accept CFTR2 agreement
print("🔐 Navigating to CFTR2 welcome page...")
driver.get("https://cftr2.org/welcome")
time.sleep(2)

checkbox_ids = ["edit-education", "edit-individual", "edit-discuss", "edit-privacy"]
for box_id in checkbox_ids:
    try:
        checkbox = wait.until(EC.presence_of_element_located((By.ID, box_id)))
        driver.execute_script("arguments[0].scrollIntoView(true);", checkbox)
        time.sleep(0.5)
        if not checkbox.is_selected():
            driver.execute_script("arguments[0].click();", checkbox)
        print(f"✅ Checked: {box_id}")
    except Exception as e:
        print(f"⚠️ Could not check box {box_id}: {e}")

try:
    submit_button = wait.until(EC.element_to_be_clickable((By.ID, "edit-submit-terms")))
    submit_button.click()
    print("✅ Submitted agreement form.")
except Exception as e:
    print(f"⚠️ Could not click submit button: {e}")

# Process each variant
failed_variants = []
variant_log = []

for variant in variant_names:
    variant_clean = variant.strip()
    if variant_clean in normalized_keys:
        print(f"⏭️ Skipping already processed variant: {variant_clean}")
        continue

    try:
        print(f"\n🌐 Processing: {variant_clean}")
        encoded_variant = double_encode(variant_clean)
        url = f"https://cftr2.org/mutation/general/{encoded_variant}"
        print(f"🔗 Encoded URL: {url}")
        driver.get(url)

        if "The website encountered an unexpected error" in driver.page_source:
            print(f"⚠️ CFTR2 error page for variant: {variant_clean}")
            failed_variants.append(variant_clean)
            variant_log.append([variant_clean, url, "CFTR2 Error Page"])
            continue

        # Click Infection tab
        infection_tab = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'infection')]"
            ))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", infection_tab)
        WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'infection')]")))
        infection_tab.click()

        # Wait for table to load
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//table[.//td[contains(text(), 'infected')]]"))
        )

        # Save HTML
        html_path = os.path.join(SAVE_FOLDER, f"{safe_filename(variant_clean)}.html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"✅ Saved: {html_path}")

        # Extract infection data
        infection_data = extract_pseudomonas_infection_from_table(html_path)
        infection_data_all[variant_clean] = infection_data
        print(f"🦠 Extracted: {infection_data}")
        variant_log.append([variant_clean, url, "Success"])

        # Save progress
        with open("cftr2_pseudomonas_infection.json", "w", encoding="utf-8") as f:
            json.dump(infection_data_all, f, indent=2)

    except Exception as e:
        print(f"❌ Error with {variant_clean}: {e}")
        failed_variants.append(variant_clean)
        variant_log.append([variant_clean, url, f"Error: {str(e)}"])
        traceback.print_exc()

driver.quit()

# Save failed variants
with open("failed_variants_infection.txt", "w") as f:
    f.write("\n".join(failed_variants))

# Save variant URL log
with open("variant_url_log_infection.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Variant", "Encoded URL", "Status"])
    writer.writerows(variant_log)

print("\n📁 All infection data saved to cftr2_pseudomonas_infection.json")
print("📄 Failed variants saved to failed_variants_infection.txt")
print("📊 Variant URL log saved to variant_url_log_infection.csv")


🔐 Visiting CFTR2 welcome page...
✅ Checked: edit-education
✅ Checked: edit-individual
✅ Checked: edit-discuss
✅ Checked: edit-privacy
✅ Agreement submitted.
🌐 Navigating to scientific page for: F508del
🛑 Closed browser session.


KeyboardInterrupt: 